# Testing the model

In [7]:
from pathlib import Path
from transformers import pipeline

BACKEND_DIR = Path.cwd().parent  # notebook is in backend/test_scripts, so go up one level
MODEL_DIR = BACKEND_DIR / "model" / "audio"
AUDIO_FILE = Path.cwd() / "test_content" / "recording2.mp3"

print(MODEL_DIR.exists())   # should be True
print(AUDIO_FILE.exists())  # should be True

detector = pipeline("audio-classification", model=str(MODEL_DIR))
result = detector(str(AUDIO_FILE))
print(result)

True
True


Loading weights: 100%|██████████| 215/215 [00:00<00:00, 383.93it/s]


[{'score': 0.9997419714927673, 'label': 'fake'}, {'score': 0.00025797123089432716, 'label': 'real'}]


In [8]:
import numpy as np

sr = detector.feature_extractor.sampling_rate

# 1. All your test files
for f in (Path.cwd() / "test_content").glob("*.mp3"):
    print(f.name, detector(str(f))[0])

# 2. Pure silence and random noise
print("silence:", detector(np.zeros(sr * 4, dtype=np.float32))[0])
print("noise:  ", detector(np.random.randn(sr * 4).astype(np.float32) * 0.1)[0])

Recording.mp3 {'score': 0.9997736811637878, 'label': 'fake'}
recording2.mp3 {'score': 0.9997419714927673, 'label': 'fake'}
recording4.mp3 {'score': 0.9997455477714539, 'label': 'fake'}
silence: {'score': 0.9995249509811401, 'label': 'fake'}
noise:   {'score': 0.9995357990264893, 'label': 'fake'}


In [1]:
import numpy as np

def stats(audio, sr):
    return {
        "seconds": len(audio) / sr,
        "peak": float(np.abs(audio).max()),
        "rms": float(np.sqrt(np.mean(audio ** 2))),
        "leading_silence": float((np.abs(audio) < 0.01).argmin() / sr),
    }

for label in [0, 1]:  # 0 = real, 1 = fake (check id2label)
    rows = [stats(x["audio"]["array"], x["audio"]["sampling_rate"])
            for x in train_dataset.filter(lambda e: e["label"] == label).select(range(100))]
    print(label, {k: round(np.mean([r[k] for r in rows]), 3) for k in rows[0]})

NameError: name 'train_dataset' is not defined